# DRS-HJ for Fused LASSO (Doppler signal)

Compares Douglas–Rachford splitting using the analytical proximal for the fused-LASSO penalty against DRS with HJ-Prox, plus a proximal-point variant.  Reproduces panels of Figures 1 and 3.

## Setup


In [ ]:
# ============================================================================
# ICML Paper - Experiment: Fused LASSO with Douglas-Rachford
# Figures: 1 & 3
# Description: Comparison of DRS (analytical), DRS-HJ, and PPM-HJ for fused LASSO
# ============================================================================

# ============================================================================
# CHUNK 1: SETUP - Algorithms, Helper Functions, and Definitions
# ============================================================================

import time
import torch
import numpy as np
import matplotlib.pyplot as plt
from hj_prox import hj_prox

# Set global seed and dtype
torch.set_default_dtype(torch.float64)
torch.manual_seed(42)
np.random.seed(42)
device = 'cpu'
EPS = 1e-5

# Plotting configuration
plt.rcParams.update({'font.size': 20})


# ============================================================================
# Helper Functions
# ============================================================================

def kth_order_diff_matrix(n: int, k: int) -> torch.Tensor:
    """
    Construct k-th order differencing matrix D of shape (n - k, n).
    For k=3: (D x)_i = x[i] - 3*x[i+1] + 3*x[i+2] - x[i+3]
    """
    # Start with identity matrix
    D = torch.eye(n, dtype=torch.float64, device=device)
    
    # Apply k differences
    for _ in range(k):
        # Create difference operator
        diff_op = torch.zeros(D.shape[0] - 1, D.shape[0], dtype=torch.float64, device=device)
        for i in range(D.shape[0] - 1):
            diff_op[i, i] = -1
            diff_op[i, i + 1] = 1
        D = diff_op @ D
    
    return D

def fused_lasso_objective(x: torch.Tensor, b: torch.Tensor, D: torch.Tensor, lambda_1: float) -> torch.Tensor:
    """
    Compute fused lasso objective: 0.5 * ||b - x||_2^2 + λ * ||D x||_1
    
    Args:
        x: signal (n, 1) or batch (batch_size, n)
        b: observations (n, 1)
        D: difference matrix (n-k, n)
        lambda_1: regularization parameter
    """
    if x.dim() == 1:
        x = x.unsqueeze(0)
    elif x.dim() == 2 and x.shape[1] == 1:
        x = x.t()  # Convert (n, 1) to (1, n)
    
    batch_size = x.shape[0]
    b_flat = b.squeeze()
    
    # Data fidelity term
    data_fid = 0.5 * torch.norm(x - b_flat.unsqueeze(0), p=2, dim=1) ** 2
    
    # Penalty term: ||Dx||_1
    Dx = x @ D.t()  # (batch_size, n-k)
    penalty = lambda_1 * torch.sum(torch.abs(Dx), dim=1)
    
    return data_fid + penalty


def compute_gradient(x: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Gradient of the smooth part: ∇f(x) = x - b"""
    return x - b


def soft_threshold(z: torch.Tensor, tau: float) -> torch.Tensor:
    """Soft thresholding operator."""
    return torch.sign(z) * torch.clamp(z.abs() - tau, min=0.0)


# ============================================================================
# Algorithm 1: Douglas-Rachford with Analytical Proximal Operators
# ============================================================================

@torch.no_grad()
def douglas_rachford_fused(
    x0: torch.Tensor,
    b: torch.Tensor,
    D: torch.Tensor,
    lambda_1: float,
    *,
    gamma: float = 1.0,
    relax: float = 1.0,
    max_iters: int = 2000,
    tol: float = 1e-6,
    verbose: bool = True,
    verbose_every: int = 50,
    precompute_cholesky: bool = True,
):
    """
    Douglas-Rachford Splitting for:
        min_x 0.5*||x - b||^2 + λ ||D x||_1
    Product-space split:
        F(x,w) = 0.5*||x-b||^2 + λ||w||_1,   C = {(x,w): w = D x}

    Args
    ----
    x0 : (n,1) initial point
    b  : (n,1) observation
    D  : ((n-k), n) k-th order difference (or any linear operator as matrix)
    lambda_1 : λ (float)
    gamma : DRS stepsize (>0), default 1.0 (any >0 converges)
    relax : relaxation (0,2], default 1.0
    max_iters : iteration cap
    tol : stopping on ||z_{k+1} - z_k||_2
    verbose : print progress
    verbose_every : print every this many iterations
    precompute_cholesky : cache Cholesky of (I + D^T D) for fast projections

    Returns
    -------
    x_star : (n,1) estimated signal
    f_hist : (T,) objective values along the run (evaluated at x_k = prox_F^x(z_k))
    diff_hist : (T,) fixed-point residual norms ||z_{k+1}-z_k||
    """
    # Ensure (n,1) shape
    if x0.dim() == 1:
        x0 = x0.unsqueeze(1)
    if b.dim() == 1:
        b = b.unsqueeze(1)

    n = b.shape[0]
    device, dtype = b.device, b.dtype
    m = D.shape[0]

    # z = (x, w); initialize w as Dx (feasible start helps but not required)
    x = x0.clone()
    w = (D @ x0).clone()

    eye_n = torch.eye(n, dtype=dtype, device=device)
    Dt = D.t()

    # Projection onto C: solve (I + D^T D) x = rhs once per iter
    A_proj = eye_n + Dt @ D
    if precompute_cholesky:
        # A_proj is SPD (I + D^T D), so Cholesky is safe
        L = torch.linalg.cholesky(A_proj)
        def solve_A(rhs):
            # Solve A_proj x = rhs via two triangular solves
            y = torch.cholesky_solve(rhs, L)
            return y
    else:
        def solve_A(rhs):
            return torch.linalg.solve(A_proj, rhs)

    f_hist = []
    diff_hist = []

    # Helpful constants for prox_F on x: prox_{γ * 0.5||·-b||^2}(v) = (v + γ b)/(1 + γ)
    inv_one_plus_gamma = 1.0 / (1.0 + gamma)

    # Work buffers
    xA = torch.empty_like(x)
    wA = torch.empty_like(w)

    for k in range(1, max_iters + 1):
        t0 = time.time()

        # ---- Prox of F at z=(x,w) ----
        # x part (closed form)
        xA.copy_( (x + gamma * b) * inv_one_plus_gamma )
        # w part (soft threshold)
        wA.copy_( soft_threshold(w, gamma * lambda_1) )

        # ---- Reflection r = 2*prox_F(z) - z ----
        rx = 2.0 * xA - x
        rw = 2.0 * wA - w

        # ---- Projection onto C: minimize ||x - rx||^2 + ||Dx - rw||^2 ----
        rhs = rx + Dt @ rw
        xB = solve_A(rhs)
        wB = D @ xB

        # ---- DRS update with relaxation ----
        new_x = x + relax * (xB - xA)
        new_w = w + relax * (wB - wA)

        # ---- Convergence stats ----
        diff = torch.linalg.norm(new_x - x).pow(2) + torch.linalg.norm(new_w - w).pow(2)
        diff = torch.sqrt(diff)
        diff_hist.append(diff.item())

        # Objective evaluated at the current primal candidate x_k = xA
        fval = 0.5 * torch.linalg.norm(xA - b).pow(2) + lambda_1 * torch.norm(D @ xA, p=1)
        f_hist.append(fval.item())

        if verbose and (k == 1 or k % verbose_every == 0):
            print(f"DRS iter {k:4d}: f={fval.item():.6f}, ||Δz||={diff.item():.3e}, time={time.time()-t0:.4f}s")

        x, w = new_x, new_w

        if diff.item() < tol:
            if verbose:
                print(f"DRS converged at iter {k} (||Δz||={diff.item():.3e})")
            break

    # The primal solution associated with a DRS fixed point is prox_F(z*). Use last xA.
    x_star = xA.clone()
    return x_star, torch.tensor(f_hist), torch.tensor(diff_hist)


# ============================================================================
# Algorithm 2: Douglas-Rachford with HJ-Prox
# ============================================================================

@torch.no_grad()
def douglas_rachford_fused_HJ_Prox(
    x0: torch.Tensor,
    b: torch.Tensor,
    D: torch.Tensor,
    lambda_1: float,
    *,
    gamma: float = 1.0,
    relax: float = 1.0,
    max_iters: int = 5000,
    num_samples_penalty: int = 100,
    tol: float = 1e-8,
    verbose: bool = True,
    verbose_every: int = 100,
    device: str = "cpu",
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    DRS for min_x 0.5||x - b||^2 + λ||Dx||_1 with inexact prox for g via HJ-Prox.
    - Delta schedule: delta = (2_500_000 * 5) / (i+1)^(2 + EPS)
    - Gamma is FIXED.
    Returns (x_hat, f_hist) where x_hat = last y_k (prox-f shadow).
    """
    # shapes
    if x0.dim() == 1: x0 = x0.unsqueeze(1)
    if b.dim()  == 1: b  = b.unsqueeze(1)
    x0 = x0.to(device); b = b.to(device); D = D.to(device)

    z = x0.clone()
    n = b.shape[0]
    b_flat = b.squeeze()
    inv_one_plus_gamma = 1.0 / (1.0 + gamma)

    # g(x) = λ ||Dx||_1   (batch-aware)
    def penalty_func(x_batch: torch.Tensor) -> torch.Tensor:
        xb = x_batch.reshape(-1, n)       # (B, n)
        Dx = xb @ D.t()                   # (B, n-k)
        return lambda_1 * torch.sum(torch.abs(Dx), dim=1)

    f_hist: list[float] = []

    for i in range(1, max_iters + 1):
        # Delta schedule
        delta = (2500000*5) / ((i + 1)**(2.0 + EPS))

        # Prox_g (inexact, HJ-Prox)
        xk, ls_iters_g = hj_prox(
            z, gamma, f=penalty_func,
            delta=delta,
            num_samples=num_samples_penalty,
            alpha=1.0
        )

        # Prox_f
        v = 2.0 * xk - z
        yk = (gamma * b_flat.unsqueeze(1) + v) * inv_one_plus_gamma

        # DRS update
        step = yk - xk
        z = z + relax * step

        # Metrics
        diff = torch.linalg.norm(step)
        fval = fused_lasso_objective(yk.t(), b, D, lambda_1).item()
        f_hist.append(fval)

        if verbose and (i == 1 or i % verbose_every == 0):
            print(f"DRS(HJ) it {i:4d}: f={fval:.6f}, ||y-x||={diff.item():.3e}, "
                  f"delta={delta:.2e}, ls_g={ls_iters_g}")

        if diff.item() < tol:
            if verbose:
                print(f"DRS(HJ) converged at it {i} (||y-x||={diff.item():.3e})")
            break

    x_hat = yk.clone()
    return x_hat, torch.tensor(f_hist)


# ============================================================================
# Algorithm 3: Proximal Point Method with HJ-Prox
# ============================================================================

def proximal_point_fused_HJ_Prox(
    x0: torch.Tensor,
    b: torch.Tensor,
    D: torch.Tensor,
    lambda_1: float,
    *,
    gamma: float = 1.0,
    max_iters: int = 5000,
    num_samples: int = 1000,
    delta_init: float = 1.0,
    tol: float = 1e-8,
    verbose: bool = True,
    verbose_every: int = 100,
    device: str = "cpu",
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Proximal Point Method for fused lasso using HJ-Prox.
    
    Iteratively computes: x^(k+1) = prox_{γF}(x^k)
    where F(x) = 0.5||x - b||^2 + λ||Dx||_1
    
    Parameters:
    -----------
    x0 : torch.Tensor
        Initial signal (n, 1) or (n,)
    b : torch.Tensor
        Observations (n, 1) or (n,)
    D : torch.Tensor
        Difference matrix (n-k, n)
    lambda_1 : float
        Regularization parameter
    gamma : float
        Fixed proximal parameter (step size)
    max_iters : int
        Maximum number of iterations
    num_samples : int
        Number of samples for HJ-Prox approximation
    delta_init : float
        Initial smoothing parameter for HJ-Prox
    tol : float
        Convergence tolerance
    verbose : bool
        Whether to print progress
    verbose_every : int
        Print frequency
    device : str
        Device to use ('cpu' or 'cuda')
        
    Returns:
    --------
    x_final : torch.Tensor
        Optimized signal (n, 1)
    f_hist : torch.Tensor
        Objective values at each iteration
    diff_hist : torch.Tensor
        Iterate differences at each iteration
    """
    # Convert inputs to proper format
    if x0.dim() == 1:
        x0 = x0.unsqueeze(1)
    if b.dim() == 1:
        b = b.unsqueeze(1)
    
    x0 = x0.to(device)
    b = b.to(device)
    D = D.to(device)
    
    n = b.shape[0]
    b_flat = b.squeeze()
    
    # Initialize
    x = x0.clone()
    
    f_hist: list[float] = []
    diff_hist: list[float] = []
    
    # Define full objective function for batched input
    def full_objective_batch(x_batch: torch.Tensor) -> torch.Tensor:
        """
        Compute full fused lasso objective for a batch of signals.
        Args:
            x_batch: shape (batch_size, n)
        Returns:
            objectives: shape (batch_size,)
        """
        # Ensure proper shape
        if x_batch.dim() == 1:
            x_batch = x_batch.unsqueeze(0)
        elif x_batch.dim() == 2 and x_batch.shape[1] == 1:
            x_batch = x_batch.t()  # Convert (n, 1) to (1, n)
        
        batch_size = x_batch.shape[0]
        
        # Data fidelity term: 0.5 * ||x - b||^2
        data_fid = 0.5 * torch.sum((x_batch - b_flat.unsqueeze(0)) ** 2, dim=1)
        
        # Penalty term: λ * ||Dx||_1
        Dx = x_batch @ D.t()  # (batch_size, n-k)
        penalty = lambda_1 * torch.sum(torch.abs(Dx), dim=1)
        
        return data_fid + penalty
    
    # Single objective for tracking (non-batched)
    def single_objective(x_vec: torch.Tensor) -> float:
        """Compute objective for a single signal vector."""
        x_flat = x_vec.squeeze()
        data_fid = 0.5 * torch.sum((x_flat - b_flat) ** 2)
        Dx = x_flat @ D.t()
        penalty = lambda_1 * torch.sum(torch.abs(Dx))
        return (data_fid + penalty).item()
    
    # Initialize objective value
    if verbose:
        initial_obj = single_objective(x)
        print("Starting Proximal Point Method with HJ-Prox for Fused Lasso...")
        print(f"λ = {lambda_1}")
        print(f"Fixed γ = {gamma}")
        print(f"HJ-Prox: num_samples = {num_samples}, delta_init = {delta_init}")
        print(f"Initial objective: {initial_obj:.6f}")
        print("-" * 70)
    
    for i in range(1, max_iters + 1):
        x_old = x.clone()
        
        # Adaptive delta schedule (annealing)
        delta = (2500000*5) / ((i + 1)**(2.0 + EPS))
        
        # Proximal Point step: x^(k+1) = prox_{γF}(x^k)
        x_new, ls_iters = hj_prox(
            x, gamma, f=full_objective_batch,
            delta=delta,
            num_samples=num_samples,
            alpha=1.0
        )
        
        x = x_new.clone()
        
        # Compute objective
        current_obj = single_objective(x)
        f_hist.append(current_obj)
        
        # Compute convergence metric
        diff = torch.linalg.norm(x - x_old).item()
        diff_hist.append(diff)
        
        if verbose and (i == 1 or i % verbose_every == 0):
            print(f"PPM(HJ) iter {i:4d}: f={current_obj:.6f}, "
                  f"||Δx||={diff:.3e}, "
                  f"δ={delta:.3e}, ls_iters={ls_iters}")
        
        # Check convergence
        if diff < tol:
            if verbose:
                print(f"\nPPM(HJ) converged at iteration {i}")
                print(f"Final objective: {current_obj:.6f}")
                print(f"Final ||Δx||: {diff:.3e}")
            break
    
    if i == max_iters and verbose:
        print(f"\nPPM(HJ) reached maximum iterations ({max_iters})")
        print(f"Final objective: {current_obj:.6f}")
    
    x_final = x.clone()
    return x_final, torch.tensor(f_hist), torch.tensor(diff_hist)


print("✓ All algorithms and helper functions loaded successfully")

## Problem definition


In [ ]:
# ============================================================================
# CHUNK 2: DATA GENERATION
# ============================================================================

def doppler(t: np.ndarray) -> np.ndarray:
    """Doppler test function (Donoho & Johnstone). Domain t ∈ (0, 1)."""
    return np.sqrt(t * (1 - t)) * np.sin((2.1 * np.pi) / (t + 0.05))


print("\n" + "="*70)
print("Generating Doppler signal data...")
print("="*70)

# Generate data
n = 256
t_vals = np.linspace(0.01, 0.99, n)
y_true = doppler(t_vals)

# Add noise
noise_level = 0.1
y_noisy = y_true + noise_level * np.random.randn(n)

# Convert to torch
b = torch.from_numpy(y_noisy).to(torch.float64).unsqueeze(1).to(device)  # shape (n, 1)

# Create the difference matrix
k_order = 3
D = kth_order_diff_matrix(n, k_order)

print(f"✓ Data generated: n={n} time points")
print(f"✓ Noise level: {noise_level}")
print(f"✓ Difference matrix: {k_order}-th order, shape {D.shape}")

# Initial point
x0 = torch.zeros((n, 1), dtype=torch.float64, device=device)

# Parameters
lambda_1 = 0.9
gamma = 1.0

print(f"✓ Regularization parameter λ = {lambda_1}")


## Algorithm 1 — DRS with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 3: RUN ALGORITHM 1 - Douglas-Rachford with HJ-Prox
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 1: Douglas-Rachford with HJ-Prox...")
print("="*70)

start_time = time.time()

x_HJ, f_hist_HJ = douglas_rachford_fused_HJ_Prox(
    x0=x0,
    b=b,
    D=D,
    lambda_1=lambda_1,
    gamma=0.000015,
    relax=1,
    max_iters=150000,
    num_samples_penalty=1000,
    tol=1e-16,
    verbose=True
)

elapsed_time = time.time() - start_time

print(f"\n✓ DRS-HJ completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(f_hist_HJ)} iterations")
print(f"  - Final objective: {f_hist_HJ[-1]:.6f}")

## Algorithm 2 — DRS with the analytical proximal


In [ ]:
# ============================================================================
# CHUNK 4: RUN ALGORITHM 2 - Analytical Douglas-Rachford
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 2: Douglas-Rachford with Analytical Proximal Operators...")
print("="*70)

start_time = time.time()

x_Analytical, f_hist_Analytical, diff_hist_Analytical = douglas_rachford_fused(
    x0=b.clone(),
    b=b,
    D=D,
    lambda_1=lambda_1,
    gamma=0.0001,
    relax=1,
    max_iters=150000,
    tol=1e-16,
    verbose=True,
    verbose_every=100
)

elapsed_time = time.time() - start_time

print(f"\n✓ DRS-Analytical completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(f_hist_Analytical)} iterations")
print(f"  - Final objective: {f_hist_Analytical[-1]:.6f}")

## Comparison: DRS vs DRS-HJ


In [ ]:
import os
os.makedirs('figures', exist_ok=True)
# ============================================================================
# CHUNK 5: GENERATE FIGURES - DRS-HJ vs Analytical DRS
# ============================================================================

print("\n" + "="*70)
print("Generating figures: DRS-HJ vs Analytical DRS comparison...")
print("="*70)

# --- Figure 1: Noisy Observations and Ground Truth ---
plt.figure(figsize=(11, 10))
plt.scatter(t_vals, y_noisy, s=30, alpha=0.5,
            label='Noisy Observations', zorder=1)
plt.plot(t_vals, y_true, 'k-', linewidth=3,
         label='True Signal', zorder=2)
plt.ylabel('Signal Value', fontsize=40)
plt.xlabel('Signal Index', fontsize=40)
plt.title('Fused LASSO Ground Truth', fontsize=40)
plt.legend(fontsize=40)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_ground_truth.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 2: DRS-HJ Solution ---
plt.figure(figsize=(11, 10))
plt.scatter(t_vals, y_noisy, s=30, alpha=0.5, zorder=1)
plt.plot(t_vals, y_true, 'k-', linewidth=3, zorder=2)
plt.plot(t_vals, x_HJ.squeeze().cpu().numpy(), 'b-', linewidth=3,
         label='DRS-HJ', alpha=0.8, zorder=4)
plt.ylabel('Signal Value', fontsize=40)
plt.xlabel('Signal Index', fontsize=40)
plt.title('DRS-HJ', fontsize=40)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_drs_hj.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 3: Analytical DRS Solution ---
plt.figure(figsize=(11, 10))
plt.scatter(t_vals, y_noisy, s=30, alpha=0.5, zorder=1)
plt.plot(t_vals, y_true, 'k-', linewidth=3, zorder=2)
plt.plot(t_vals, x_Analytical.squeeze().cpu().numpy(), 'r--', linewidth=3,
         label='DRS', alpha=0.8, zorder=3)
plt.ylabel('Signal Value', fontsize=40)
plt.xlabel('Signal Index', fontsize=40)
plt.title('DRS', fontsize=40)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_coordinate_descent.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 4: Objective Function Convergence (DRS-HJ vs Analytical) ---
plt.figure(figsize=(11, 10))
plt.semilogy(f_hist_HJ.numpy(), '-', linewidth=3,
             label=f'DRS-HJ: {f_hist_HJ[-1].item():.3f}')
plt.semilogy(f_hist_Analytical.numpy(), '--', linewidth=3,
             label=f'DRS: {f_hist_Analytical[-1].item():.3f}')
plt.ylabel('Objective (log scale)', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.title('DRS Convergence', fontsize=40)
plt.legend(fontsize=40, loc='upper left')
plt.grid(True, alpha=0.3, which='both')
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_convergence.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: fused_lasso_ground_truth.pdf, fused_lasso_drs_hj.pdf, fused_lasso_coordinate_descent.pdf, fused_lasso_convergence.pdf")


## Algorithm 3 — Proximal-point method with HJ-Prox


In [ ]:
# ============================================================================
# CHUNK 6: RUN ALGORITHM 3 - Proximal Point Method with HJ-Prox
# ============================================================================

print("\n" + "="*70)
print("Running Algorithm 3: Proximal Point Method with HJ-Prox...")
print("="*70)

start_time = time.time()

x_PPM, f_PPM, diff_PPM = proximal_point_fused_HJ_Prox(
    x0, b, D, lambda_1,
    gamma=0.000005,
    max_iters=150000,
    num_samples=1000,
    delta_init=1.0,
    tol=1e-18,
    verbose=True,
    verbose_every=100,
    device=device
)

elapsed_time = time.time() - start_time

print(f"\n✓ PPM-HJ completed")
print(f"  - Runtime: {elapsed_time:.2f} seconds")
print(f"  - Converged in {len(f_PPM)} iterations")
print(f"  - Final objective: {f_PPM[-1]:.6f}")

## Comparison: DRS-HJ vs PPM-HJ


In [ ]:
# ============================================================================
# CHUNK 7: GENERATE FIGURES - DRS-HJ vs PPM-HJ
# ============================================================================

print("\n" + "="*70)
print("Generating figures: DRS-HJ vs PPM-HJ comparison...")
print("="*70)

# --- Figure 5: Ground Truth (repeated for this comparison set) ---
plt.figure(figsize=(11, 10))
plt.scatter(t_vals, y_noisy, s=30, alpha=0.5,
            label='Noisy Observations', zorder=1)
plt.plot(t_vals, y_true, 'k-', linewidth=3,
         label='True Signal', zorder=2)
plt.ylabel('Signal Value', fontsize=40)
plt.xlabel('Signal Index', fontsize=40)
plt.title('Fused LASSO Ground Truth', fontsize=40)
plt.legend(fontsize=40)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_ground_truth.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 6: DRS-HJ Solution (repeated) ---
plt.figure(figsize=(11, 10))
plt.scatter(t_vals, y_noisy, s=30, alpha=0.5, zorder=1)
plt.plot(t_vals, y_true, 'k-', linewidth=3, zorder=2)
plt.plot(t_vals, x_HJ.squeeze().cpu().numpy(), 'b-', linewidth=3,
         label='DRS-HJ', alpha=0.8, zorder=4)
plt.ylabel('Signal Value', fontsize=40)
plt.xlabel('Signal Index', fontsize=40)
plt.title('DRS-HJ', fontsize=40)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_drs_hj.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 7: PPM-HJ Solution ---
plt.figure(figsize=(11, 10))
plt.scatter(t_vals, y_noisy, s=30, alpha=0.5, zorder=1)
plt.plot(t_vals, y_true, 'k-', linewidth=3, zorder=2)
plt.plot(t_vals, x_PPM.squeeze().cpu().numpy(), 'r--', linewidth=3,
         label='PPM-HJ', alpha=0.8, zorder=3)
plt.ylabel('Signal Value', fontsize=40)
plt.xlabel('Signal Index', fontsize=40)
plt.title('PPM-HJ', fontsize=40)
plt.grid(True, alpha=0.3)
plt.xlim(0, 1)
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_PPM.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

# --- Figure 8: Objective Function Convergence (DRS-HJ vs PPM-HJ) ---
plt.figure(figsize=(11, 10))
plt.semilogy(f_hist_HJ.numpy(), '-', linewidth=3,
             label=f'DRS-HJ: {f_hist_HJ[-1].item():.3f}')
plt.semilogy(f_PPM.numpy(), '--', linewidth=3,
             label=f'PPM-HJ: {f_PPM[-1].item():.3f}')
plt.ylabel('Objective (log scale)', fontsize=40)
plt.xlabel('Iteration', fontsize=40)
plt.title('Fused LASSO', fontsize=40)
plt.legend(fontsize=40, loc='upper left')
plt.grid(True, alpha=0.3, which='both')
plt.tick_params(axis='both', which='major', labelsize=40)
plt.tight_layout()
plt.savefig('figures/fused_lasso_convergence_VSPPM.pdf', format='pdf', dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print("✓ All figures saved: fused_lasso_PPM.pdf, fused_lasso_convergence_VSPPM.pdf")
print("\n" + "="*70)
print("✓ ALL EXPERIMENTS COMPLETED SUCCESSFULLY")
print("="*70)